https://github.com/langchain-ai/langchain/tree/master/libs/langchain/langchain/retrievers/document_compressors

https://blog.langchain.dev/improving-document-retrieval-with-contextual-compression/

In [ ]:
!pip install langchain_community

In [ ]:
#facebook ai similarity search
!pip install faiss-cpu

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

In [2]:
documents = TextLoader(
    r"F:\Advance RAG\Data\state_of_the_union.txt",
    encoding="utf-8"
).load()

In [3]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [4]:
texts = text_splitter.split_documents(documents)

In [5]:
type(texts)

list

In [6]:
len(texts)

95

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [9]:
retriever = FAISS.from_documents(texts, embeddings).as_retriever()

In [10]:
docs = retriever.invoke("What did the president say about Ketanji Brown Jackson")

In [11]:
# Helper function for printing docs

def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [12]:
pretty_print_docs(docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
----------------------------------------------------------------------------------------------------
Document 2:

We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service.
----------------------------------------------------------

In [13]:
docs2 = retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [14]:
pretty_print_docs(docs2)

Document 1:

Because I see the future that is within our grasp. 

Because I know there is simply nothing beyond our capacity. 

We are the only nation on Earth that has always turned every crisis we have faced into an opportunity. 

The only nation that can be defined by a single word: possibilities. 

So on this night, in our 245th year as a nation, I have come to report on the State of the Union. 

And my report is this: the State of the Union is strong—because you, the American people, are strong.
----------------------------------------------------------------------------------------------------
Document 2:

Third – we can end the shutdown of schools and businesses. We have the tools we need. 

It’s time for Americans to get back to work and fill our great downtowns again.  People working from home can feel safe to begin to return to the office.   

We’re doing that here in the federal government. The vast majority of federal workers will once again work in person. 

Our schools ar

In [15]:
docs3 = retriever.invoke("How did the President propose to tackle the issue of climate change?")


In [16]:
pretty_print_docs(docs3)

Document 1:

Second – cut energy costs for families an average of $500 a year by combatting climate change.  

Let’s provide investments and tax credits to weatherize your homes and businesses to be energy efficient and you get a tax credit; double America’s clean energy production in solar, wind, and so much more;  lower the price of electric vehicles, saving you another $80 a month because you’ll never have to pay at the gas pump again.
----------------------------------------------------------------------------------------------------
Document 2:

My plan to fight inflation will lower your costs and lower the deficit. 

17 Nobel laureates in economics say my plan will ease long-term inflationary pressures. Top business leaders and most Americans support my plan. And here’s the plan: 

First – cut the cost of prescription drugs. Just look at insulin. One in ten Americans has diabetes. In Virginia, I met a 13-year-old boy named Joshua Davis.
-------------------------------------------

In [ ]:
! pip install -U langchain-groq

In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [18]:
from langchain_groq import ChatGroq

In [19]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

In [20]:
result = llm.invoke("What is RAG?")

print(result.content)

RAG can refer to different things depending on the context. Here are a few possible meanings:

1. **RAG (Red, Amber, Green) status**: In project management, RAG is a color-coded system used to indicate the status of a project or task. Red typically indicates a problem or high risk, Amber (or Yellow) indicates a warning or caution, and Green indicates that everything is on track.
2. **RAG (Raise and Give) week**: In the UK, RAG week is a charity fundraising event organized by students, often at universities. The goal is to raise money for local charities through various activities and events.
3. **RAG (Rough As Guts)**: In some countries, including Australia and New Zealand, RAG is a colloquialism used to describe something or someone that is rough, tough, or unrefined.
4. **RAG (other meanings)**: RAG can also be an acronym for other organizations, companies, or concepts, such as RAG (Rheinisch-Westfälisches Elektrizitätswerk), a German energy company, or RAG (Reactive Airborne Generat

In [21]:
from langchain.chains import RetrievalQA

In [22]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [23]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [24]:
chain.invoke(query)

{'query': 'What were the top three priorities outlined in the most recent State of the Union address?',
 'result': 'The provided text does not explicitly state the top three priorities, but it does mention a "Unity Agenda for the Nation" with four big things that can be done together. The four items mentioned are:\n\n1. Beat the opioid epidemic\n2. (The other three items are not mentioned in the provided text, only the first one is)\n\nAdditionally, the text mentions several other initiatives and proposals, such as ending the shutdown of schools and businesses, passing the Paycheck Fairness Act, raising the minimum wage, and strengthening the Violence Against Women Act. However, it does not clearly outline the top three priorities.'}

In [25]:
print(chain.invoke(query)['result'])

The provided text does not explicitly state the top three priorities, but it does mention a "Unity Agenda for the Nation" with four big things that can be done together. The four items mentioned are:

1. Beat the opioid epidemic
2. (The other three items are not mentioned in the provided text, but the first item of the Unity Agenda is the opioid epidemic)

Additionally, the text mentions several other initiatives, such as:

* Ending the shutdown of schools and businesses
* Passing the Paycheck Fairness Act and paid leave
* Raising the minimum wage to $15 an hour
* Increasing Pell Grants and support for HBCUs and community colleges

However, without more context, it's difficult to determine the top three priorities. The provided text appears to be a partial transcript of a State of the Union address, and it may not include all the information necessary to answer the question.


# Using ContextualCompressionRetriever 

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [28]:
compressor = LLMChainExtractor.from_llm(llm)

In [29]:
compression_retriever=ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

In [30]:
compressed_docs = compression_retriever.invoke("What did the president say about Ketanji Jackson Brown")

In [31]:
compressed_docs

[Document(metadata={'source': 'F:\\Advance RAG\\Data\\state_of_the_union.txt'}, page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.')]

In [32]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [33]:
compressed_docs

[Document(metadata={'source': 'F:\\Advance RAG\\Data\\state_of_the_union.txt'}, page_content='Let’s pass the Paycheck Fairness Act and paid leave.  \nRaise the minimum wage to $15 an hour and extend the Child Tax Credit, so no one has to raise a family in poverty. \nLet’s increase Pell Grants and increase our historic support of HBCUs, and invest in what Jill—our First Lady who teaches full-time—calls America’s best-kept secret: community colleges. \nAnd let’s pass the PRO Act when a majority of workers want to form a union—they shouldn’t be stopped.'),
 Document(metadata={'source': 'F:\\Advance RAG\\Data\\state_of_the_union.txt'}, page_content='First, beat the opioid epidemic. \nSo tonight I’m offering a Unity Agenda for the Nation. Four big things we can do together.  \nFirst, beat the opioid epidemic. \nThere is so much we can do. Increase funding for prevention, treatment, harm reduction, and recovery.')]

In [34]:
len(compressed_docs)

2

In [35]:
pretty_print_docs(compressed_docs)

Document 1:

Let’s pass the Paycheck Fairness Act and paid leave.  
Raise the minimum wage to $15 an hour and extend the Child Tax Credit, so no one has to raise a family in poverty. 
Let’s increase Pell Grants and increase our historic support of HBCUs, and invest in what Jill—our First Lady who teaches full-time—calls America’s best-kept secret: community colleges. 
And let’s pass the PRO Act when a majority of workers want to form a union—they shouldn’t be stopped.
----------------------------------------------------------------------------------------------------
Document 2:

First, beat the opioid epidemic. 
So tonight I’m offering a Unity Agenda for the Nation. Four big things we can do together.  
First, beat the opioid epidemic. 
There is so much we can do. Increase funding for prevention, treatment, harm reduction, and recovery.


In [36]:
compressed_docs2 = compression_retriever.invoke("How did the President propose to tackle the issue of climate change?")

In [37]:
pretty_print_docs(compressed_docs2)

Document 1:

Second – cut energy costs for families an average of $500 a year by combatting climate change.  
Let’s provide investments and tax credits to weatherize your homes and businesses to be energy efficient and you get a tax credit; double America’s clean energy production in solar, wind, and so much more;  lower the price of electric vehicles, saving you another $80 a month because you’ll never have to pay at the gas pump again.


# By using LLMChainFilter

In [38]:
from langchain.retrievers.document_compressors import LLMChainFilter

In [39]:
filter = LLMChainFilter.from_llm(llm)

In [41]:
compression_retriever2 = ContextualCompressionRetriever(base_compressor=filter, base_retriever=retriever)

In [42]:
compressed_docs3 = compression_retriever2.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [43]:
pretty_print_docs(compressed_docs3)

Document 1:

Third – we can end the shutdown of schools and businesses. We have the tools we need. 

It’s time for Americans to get back to work and fill our great downtowns again.  People working from home can feel safe to begin to return to the office.   

We’re doing that here in the federal government. The vast majority of federal workers will once again work in person. 

Our schools are open. Let’s keep it that way. Our kids need to be in school.
----------------------------------------------------------------------------------------------------
Document 2:

Let’s pass the Paycheck Fairness Act and paid leave.  

Raise the minimum wage to $15 an hour and extend the Child Tax Credit, so no one has to raise a family in poverty. 

Let’s increase Pell Grants and increase our historic support of HBCUs, and invest in what Jill—our First Lady who teaches full-time—calls America’s best-kept secret: community colleges. 

And let’s pass the PRO Act when a majority of workers want to form a 

In [44]:
original_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(docs2)]))

In [45]:
original_contexts_len

1817

In [46]:
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

In [47]:
compressed_contexts_len

723

In [48]:
print("Original context length:", original_contexts_len)

Original context length: 1817


In [49]:
print("Compressed context length:", compressed_contexts_len)

Compressed context length: 723


In [50]:
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

Compressed Ratio: 2.51x


In [51]:
from langchain.retrievers.document_compressors import EmbeddingsFilter

In [53]:
from langchain_huggingface import HuggingFaceEmbeddings

In [54]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [96]:
embeddings_filter = EmbeddingsFilter(embeddings=embeddings)

In [97]:
compression_retriever3 = ContextualCompressionRetriever(base_compressor=embeddings_filter, base_retriever=retriever)

In [98]:
compressed_docs4 = compression_retriever3.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [100]:
pretty_print_docs(compressed_docs4)

Document 1:

Because I see the future that is within our grasp. 

Because I know there is simply nothing beyond our capacity. 

We are the only nation on Earth that has always turned every crisis we have faced into an opportunity. 

The only nation that can be defined by a single word: possibilities. 

So on this night, in our 245th year as a nation, I have come to report on the State of the Union. 

And my report is this: the State of the Union is strong—because you, the American people, are strong.
----------------------------------------------------------------------------------------------------
Document 2:

Third – we can end the shutdown of schools and businesses. We have the tools we need. 

It’s time for Americans to get back to work and fill our great downtowns again.  People working from home can feel safe to begin to return to the office.   

We’re doing that here in the federal government. The vast majority of federal workers will once again work in person. 

Our schools ar

In [59]:
print("Original context length:", original_contexts_len)

Original context length: 1817


In [60]:
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

In [61]:
print("Compressed context length:", compressed_contexts_len)

Compressed context length: 723


In [62]:
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

Compressed Ratio: 2.51x


# Using DocumentCompressorPipeline

In [86]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter


In [113]:
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=0, separator=". ")

In [114]:
type(splitter)

langchain_text_splitters.character.CharacterTextSplitter

In [137]:
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

In [138]:
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings,
    k=5
)

In [139]:
pipeline_compressor = DocumentCompressorPipeline(transformers=[splitter, redundant_filter, relevant_filter])

In [140]:
compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline_compressor, base_retriever=retriever)

In [141]:
compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

In [142]:
print(len(compressed_docs))

5


In [143]:
pretty_print_docs(compressed_docs)

Document 1:

So on this night, in our 245th year as a nation, I have come to report on the State of the Union. 

And my report is this: the State of the Union is strong—because you, the American people, are strong.
----------------------------------------------------------------------------------------------------
Document 2:

Third – we can end the shutdown of schools and businesses. We have the tools we need. 

It’s time for Americans to get back to work and fill our great downtowns again.  People working from home can feel safe to begin to return to the office.   

We’re doing that here in the federal government
----------------------------------------------------------------------------------------------------
Document 3:

Let’s increase Pell Grants and increase our historic support of HBCUs, and invest in what Jill—our First Lady who teaches full-time—calls America’s best-kept secret: community colleges. 

And let’s pass the PRO Act when a majority of workers want to form a union—

# Making Chain to connect with LLM

In [144]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0,
)

In [145]:
from langchain.chains import RetrievalQA

In [146]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)

In [147]:
query="What were the top three priorities outlined in the most recent State of the Union address?"

In [148]:
chain.invoke(query)

{'query': 'What were the top three priorities outlined in the most recent State of the Union address?',
 'result': 'The provided context does not explicitly state the top three priorities, but it does mention a "Unity Agenda for the Nation" with four big things that can be done together. The first one mentioned is beating the opioid epidemic. The other three are not explicitly stated in the provided context as the top priorities, but rather as various initiatives and goals, such as ending the shutdown of schools and businesses, increasing support for education, and strengthening the Violence Against Women Act. I don\'t know the top three priorities as they are not clearly outlined in the given context.'}

In [149]:
print(chain.invoke(query)['result'])

The provided context does not explicitly state the top three priorities, but it does mention a "Unity Agenda for the Nation" with four big things that can be done together. The first one mentioned is to "beat the opioid epidemic" by increasing funding for prevention, treatment, harm reduction, and recovery. The other three items in the Unity Agenda are not specified in the given context. Additionally, there are other topics mentioned, such as ending the shutdown of schools and businesses, increasing support for education, and passing the PRO Act, but it's unclear if these are the top priorities. I don't have enough information to provide a definitive answer.


The provided context does not explicitly state the top three priorities, but it
does mention a "Unity Agenda for the Nation" with four big things to do
together, and also mentions other initiatives. However, based on the given
information, some of the priorities mentioned include:  
1. Ending the shutdown
of schools and businesses and getting Americans back to work. 
2. Investing in education, such as increasing Pell Grants, supporting HBCUs, and community
colleges. 
3. Beating the opioid epidemic by increasing funding for prevention,
treatment, harm reduction, and recovery.  
